# Module 1 · Lab 1.1 — Prompting Techniques & Structured Outputs
## Five prompting patterns, one worked banking example each — ending in output a system can consume

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---

**Scenario.** A retail bank's servicing team handles thousands of free-text customer messages a day —
loan enquiries, failed-transaction complaints, trade instructions. Today each one is read by a person and
retyped into a case system. This lab builds the prompt layer that classifies, summarises and structures
those messages, and ends with output the case system can consume directly rather than prose a human must
re-key.

**Learning objectives.** By the end of this lab you can:
1. Build prompts with `ChatPromptTemplate` — system and human messages, template variables and a
   `MessagesPlaceholder` for conversation history.
2. Count tokens, and explain what a context window limits and what it costs.
3. Apply **zero-shot**, **few-shot**, **directional stimulus**, **Chain-of-Thought** and
   **Tree-of-Thought** prompting — and say which pattern a task calls for.
4. Constrain model output to a validated **Pydantic** schema, and handle the case where it does not match.
5. Say when a raw provider SDK is enough and when LangChain earns its place.

> 🗺️ **Workshop run-list.**
> **Live in session:** §1–11 — set-up, messages and templates, tokens, the five patterns (Tree-of-Thought is the designated cut if time is short), structured output ⭐, two applied scenarios, choosing a pattern, §13
> **After the session:** §12 challenge — kept in full so you can finish at your own pace (marked ⏭️).

## 1. Set-up

In [1]:
# 📦 Packages this notebook needs. On the workshop VM they are pre-installed from
# the workshop's requirements.txt, so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU langchain langchain-core langchain-openai openai pydantic python-dotenv tiktoken

from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_PACKAGES = [
    "langchain",
    "langchain-core",
    "langchain-openai",
    "openai",
    "pydantic",
    "python-dotenv",
    "tiktoken",
]
for package_name in NOTEBOOK_PACKAGES:
    try:
        print(f"{package_name:<26} {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name:<26} NOT INSTALLED — uncomment the !pip line above")

langchain                  1.4.3
langchain-core             1.6.6
langchain-openai           1.6.6
openai                     2.54.0
pydantic                   2.12.5
python-dotenv              1.2.3
tiktoken                   0.14.0


In [2]:
import os

# Where to create each key. The trainer supplies OPENAI_API_KEY on the workshop VM;
# you create any other key yourself and add it to the workshop's .env file as KEY_NAME=value.
KEY_SIGNUP_PAGES = {
    "OPENAI_API_KEY": "supplied on the workshop VM (elsewhere: https://platform.openai.com/api-keys)",
    "TAVILY_API_KEY": "https://app.tavily.com (free tier)",
    "GROQ_API_KEY": "https://console.groq.com/keys (free tier)",
    "WEATHER_API_KEY": "https://www.weatherapi.com/signup.aspx (free tier)",
    "LANGSMITH_API_KEY": "https://smith.langchain.com -> Settings -> API Keys (free tier)",
    "ANTHROPIC_API_KEY": "https://console.anthropic.com/settings/keys",
    "GEMINI_API_KEY": "https://aistudio.google.com/app/apikey",
}


def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere).

    Prints one line per key, then stops with instructions if a required key is missing.

    Args:
        required: the environment-variable names this lab needs.
    """
    try:
        from google.colab import userdata          # running on Google Colab
        for key_name in required:
            try:
                os.environ[key_name] = userdata.get(key_name)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # the workshop VM, or any local machine
        from dotenv import find_dotenv, load_dotenv
        load_dotenv(find_dotenv(usecwd=True))        # searches upward for the workshop's .env file
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for key_name in required:
        print(f"   {key_name:<22} {'✅' if os.environ.get(key_name) else '❌ MISSING'}")
    missing_keys = [key_name for key_name in required if not os.environ.get(key_name)]
    if missing_keys:
        instructions = "\n".join(
            f"  {key_name}: {KEY_SIGNUP_PAGES.get(key_name, 'see README.md in the workshop folder')}"
            for key_name in missing_keys)
        raise RuntimeError(
            "This lab needs API keys that are not set yet. Create each one, add it to\n"
            "the workshop's .env file as KEY_NAME=value, then re-run this cell:\n" + instructions)


load_keys(["OPENAI_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅


In [3]:
from typing import List, Literal

from IPython.display import Markdown, display
from langchain_core.exceptions import OutputParserException
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, ValidationError

# One model for most of the lab; temperature 0.1 keeps the side-by-side comparisons repeatable.
chat_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
# The reasoning-pattern examples (Chain-of-Thought, Tree-of-Thought) ran a little warmer at source.
exploring_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)


def run_prompt(prompt_text, title, model=None, **variables):
    """Run a prompt as ChatPromptTemplate | model | StrOutputParser() and display the reply.

    Args:
        prompt_text: the prompt, with {placeholders} filled from `variables`.
        title: a label printed above the reply.
        model: the chat model to use; defaults to chat_model.
        **variables: values for the prompt's placeholders.

    Returns:
        The model's reply as plain text.
    """
    prompt = ChatPromptTemplate.from_template(prompt_text)
    chain = prompt | (model or chat_model) | StrOutputParser()
    reply = chain.invoke(variables)
    print(f"--- {title} ---")
    display(Markdown(reply))
    return reply


print("Models ready: gpt-4.1-mini (temperature 0.1 and 0.3)")

Models ready: gpt-4.1-mini (temperature 0.1 and 0.3)


## 2. Chat models, message types and `ChatPromptTemplate`

A chat model does not take "a prompt" — it takes a **list of messages**, each with a role:

| Message | Who says it | Typical use |
|---|---|---|
| `SystemMessage` | you, the developer | role, rules, output format — the instructions that frame every turn |
| `HumanMessage` | the user | the question or the document to work on |
| `AIMessage` | the model | its earlier replies, when you send a conversation back |
| `ToolMessage` | a tool | the result of a tool call — you meet it in Module 5 |

`ChatPromptTemplate` builds that list for you from a template plus variables, and
`MessagesPlaceholder` reserves a slot where a whole conversation can be dropped in.

In [4]:
# The raw form: a list of typed messages.
conversation = [
    SystemMessage(content="You are a retail-banking assistant. Answer in one sentence."),
    HumanMessage(content="What does a credit score tell a lender?"),
]
reply = chat_model.invoke(conversation)
print(type(reply).__name__, "->", reply.content)
print("Token usage for this call:", reply.usage_metadata)

AIMessage -> A credit score tells a lender how likely you are to repay borrowed money based on your credit history.
Token usage for this call: {'input_tokens': 33, 'output_tokens': 20, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [5]:
# The template form: roles fixed, content filled in per call, and a slot for history.
servicing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} at a retail bank. Answer in at most {sentence_limit} sentences."),
    MessagesPlaceholder("conversation_history"),
    ("human", "{customer_message}"),
])

prompt_values = {
    "role": "customer-service assistant",
    "sentence_limit": 2,
    "conversation_history": [
        HumanMessage(content="My card was declined at a fuel station this morning."),
        AIMessage(content="Sorry to hear that. Was it a debit card or a credit card?"),
    ],
    "customer_message": "Debit card. What should I check first?",
}

# See exactly what the model will receive...
for message in servicing_prompt.invoke(prompt_values).to_messages():
    print(f"{message.type:>6}: {message.content}")

# ...then run it as a chain: prompt | model | parser.
servicing_chain = servicing_prompt | chat_model | StrOutputParser()
print("\nReply:", servicing_chain.invoke(prompt_values))

system: You are a customer-service assistant at a retail bank. Answer in at most 2 sentences.
 human: My card was declined at a fuel station this morning.
    ai: Sorry to hear that. Was it a debit card or a credit card?
 human: Debit card. What should I check first?



Reply: Please check if your account has sufficient funds and ensure your card is not blocked or expired. If everything seems fine, contact our customer service for further assistance.


### LangChain, or just the provider's SDK?

For **one call to one provider**, the raw SDK is enough — you will see the OpenAI SDK do structured output
in section 9 in a handful of lines. LangChain earns its place when you need what a single call does not
give you: **composition** (`prompt | model | parser`), **swapping providers** without rewriting code,
**retries and fallbacks** (Lab 2.2), **tracing** in LangSmith, and plugging the same pieces into
**retrieval** (Modules 3–4) and **agents** (Module 5 onward).

## 3. Tokens & context windows — the units a model actually sees

LLMs don't read characters or words — they read **tokens** (chunks of ~4 characters of English on average). Two consequences you must internalize:

1. **You pay per token** (input + output). Cost and latency scale with token count.
2. A model can only "see" a fixed number of tokens at once — its **context window**. Go past it and the earliest text is dropped.

Let's look at real tokens using OpenAI's `tiktoken`. (Each provider uses a *different* tokenizer, so token counts won't match exactly across providers — keep that in mind when comparing costs.)

In [6]:
import tiktoken

token_encoder = tiktoken.get_encoding("o200k_base")   # tokenizer family used by recent OpenAI models
sample_text = "ShopKart's Christmas sale starts Friday! Up to 60% off. 🎉"

token_ids = token_encoder.encode(sample_text)
print(f"Text        : {sample_text}")
print(f"Characters  : {len(sample_text)}")
print(f"Tokens      : {len(token_ids)}")
print(f"Token IDs   : {token_ids}")
print(f"Each token  : {[token_encoder.decode([token_id]) for token_id in token_ids]}")

Text        : ShopKart's Christmas sale starts Friday! Up to 60% off. 🎉
Characters  : 57
Tokens      : 17
Token IDs   : [19083, 179367, 885, 10431, 7357, 13217, 9377, 0, 4686, 316, 220, 1910, 4, 1277, 13, 139786, 231]
Each token  : ['Shop', 'Kart', "'s", ' Christmas', ' sale', ' starts', ' Friday', '!', ' Up', ' to', ' ', '60', '%', ' off', '.', ' �', '�']


**Read the output above:** notice how common words become single tokens, while emoji, rare words, and punctuation can split into several. Now compare the headline context windows (approximate — verify current values):

| Model | Context window (tokens) | Rough equivalent |
|---|---|---|
| `gpt-4.1-mini` | ~1,000,000 | a long book |
| `gemini-2.5-flash` | ~1,000,000 | a long book |
| `claude-haiku-4-5` | ~200,000 | a short book |

> **Business takeaway:** "Can I just paste the whole 300-page contract into the prompt?" Sometimes yes —
> but you pay for every token, every time. That is *why* retrieval (Module 3–4)
> exists, and why section 2's `usage_metadata` is worth printing on every call you care about.

## 4. Zero-shot — classify a customer request

**Background:** This pattern involves providing the model with examples ("shots") to guide its output.

  * **Zero-Shot:** You provide no examples, just the instruction. This works for simple, well-defined tasks.
  * **Few-Shot:** You provide a few examples of input and desired output. This is powerful for teaching the model a specific format, structure, or nuanced classification.

**Scenario:** A banking chatbot receives an incoming customer message. It needs to classify the message into a predefined category for routing to the correct department.

In [7]:
customer_message = "Hi, my credit card was charged twice for a purchase I made at BigBasket yesterday. Can you please help me fix this?"

zeroshot_prompt = """
You are a classification bot.
Classify the following customer message into one of these categories:
- Account Inquiry
- Card Dispute
- Loan Application
- Technical Support

Return only the category name.

Message: "{message}"
"""

_ = run_prompt(zeroshot_prompt, "Zero-shot classification", message=customer_message)

--- Zero-shot classification ---


Card Dispute

*Critique:* This is a perfect use of zero-shot prompting. The task is clear, and the model can easily perform it without needing examples.

## 5. Few-shot — structure a trader's chat message

**Scenario:** A system needs to parse unstructured text from a trader's chat message and convert it into a structured format for the order management system. The format must be exact.

#### **Naive Prompt 👎**

A naive prompt will result in an unpredictable structure. The keys and values might change with every run.

In [8]:
trader_chat = "we need to buy 5000 shares of RELIANCE at market price asap"

naive_fewshot_prompt = "Extract the details from this trade message: {message}"
_ = run_prompt(naive_fewshot_prompt, "Naive trade extraction", message=trader_chat)

--- Naive trade extraction ---


Here are the extracted details from the trade message:

- Action: Buy  
- Quantity: 5000 shares  
- Stock: RELIANCE  
- Price: Market price  
- Urgency: As soon as possible (ASAP)

*Critique:* This is just a string, not a machine-readable format. It's also missing a key for the price.

#### **Few-Shot Pattern Prompt 👍**

We provide a clear example to teach the model the **exact output format** we need.

In [9]:
fewshot_prompt = """
You are an AI that converts unstructured trader chat messages into a structured string.

Follow the format of the example precisely.
---
**Example:**
Message: "sell 200 INFY limit 1500"
Output: "ACTION:SELL|TICKER:INFY|QUANTITY:200|PRICE:1500.00"
---

**New Message:**
Message: "{message}"
Output:
"""

_ = run_prompt(fewshot_prompt, "Few-shot trade extraction", message=trader_chat)

--- Few-shot trade extraction ---


Output: "ACTION:BUY|TICKER:RELIANCE|QUANTITY:5000|PRICE:MARKET"

*Critique:* Perfect. The few-shot example forced the model to produce an output that is structured, consistent, and ready for a downstream system to parse.

## 6. Directional stimulus — point a credit report at its risks

**Background:** This pattern guides the LLM to focus on specific aspects of a large piece of text. Instead of a generic summary, you provide "hints" or "directions" to tailor the output to your specific needs. 🎯

**Scenario:** A busy loan officer needs to quickly understand the key risks in a lengthy business credit report. They don't need a full summary, only the negative points.

In [10]:
credit_report_excerpt = """
Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY.
Their cash flow is positive, and they hold significant assets.
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers.
The market outlook for their sector is stable, but dependent on volatile international supply chains.
"""

naive_directional_prompt = "Summarize this credit report excerpt:\n\n{report}"
_ = run_prompt(naive_directional_prompt, "Naive report summary", report=credit_report_excerpt)

--- Naive report summary ---


Future Gadgets Ltd. demonstrates consistent 15% year-over-year revenue growth with positive cash flow and substantial assets. However, the company has a high debt-to-equity ratio of 2.5 and has recently made two late payments to major suppliers. The sector's market outlook is stable but influenced by volatile international supply chains.

*Critique:* The summary is accurate but mixes positive and negative points.

#### **Directional Stimulus Prompt 👍**

We give the model a clear "hint" to focus *only* on the risk factors.

In [11]:
directional_prompt = """
Summarize the following credit report excerpt in 2-3 bullet points.

Hint: Focus exclusively on the negative points and potential risk factors mentioned in the report.

Report:
{report}
"""

_ = run_prompt(directional_prompt, "Directional risk summary", report=credit_report_excerpt)

--- Directional risk summary ---


- High debt-to-equity ratio of 2.5 indicates elevated financial leverage and potential risk.
- Two recent late payments to major suppliers suggest possible cash flow or management issues.
- Dependence on volatile international supply chains poses a risk to business stability despite a stable market outlook.

*Critique:* This is much more effective. It provides the loan officer with a concise, focused list of the exact information they need, saving them time and improving efficiency.

## 7. Chain-of-Thought — an auditable loan assessment

**Chain-of-Thought (CoT) prompting** is a technique that encourages the model to break down a complex problem into intermediate steps before giving a final answer. By simply adding phrases like "Let's think step by step," you can significantly improve performance on tasks that require logic, reasoning, or calculation.

  * **How it works:** Instead of jumping to a conclusion, the model first "thinks" out loud, detailing its reasoning process. This mimics how humans solve complex problems and often leads the model to a more accurate result.
  * **Strengths:** Massively improves accuracy on arithmetic, commonsense, and symbolic reasoning tasks. The output is more transparent and auditable, as you can see *how* the model reached its conclusion.

**Scenario:** A loan officer needs a quick, reasoned assessment of a small business loan application to decide if it warrants a full underwriting process.

In [12]:
loan_application_data = """
- Business Name: 'Global Exports Inc.'
- Years in Business: 2
- Annual Revenue: ₹75,00,000
- Requested Loan Amount: ₹20,00,000
- Owner's CIBIL Score: 680
---
Bank's Minimum Criteria:
- Minimum Years in Business: 3
- Minimum Annual Revenue: ₹50,00,000
- Minimum CIBIL Score: 720
"""

naive_loan_prompt = "Based on the data and criteria, should we approve a loan for Global Exports Inc.? \n\n{data}"
_ = run_prompt(naive_loan_prompt, "Naive loan assessment", model=exploring_model,
               data=loan_application_data)

--- Naive loan assessment ---


Based on the provided data and the bank's minimum criteria:

- **Years in Business:** Global Exports Inc. has been in business for 2 years, but the minimum required is 3 years. **Does not meet criteria.**
- **Annual Revenue:** ₹75,00,000, which is above the minimum required ₹50,00,000. **Meets criteria.**
- **Owner's CIBIL Score:** 680, which is below the minimum required 720. **Does not meet criteria.**

**Conclusion:** Global Exports Inc. does not meet the minimum criteria for years in business and CIBIL score. Therefore, based on the bank's criteria, the loan should **not** be approved.

*Critique:* The answer is correct, but it lacks a structured breakdown of the decision process.

#### **Improved CoT Prompt 👍**

This prompt forces the model to perform a step-by-step analysis, comparing each application data point against the criteria before reaching a conclusion.

In [13]:
cot_loan_prompt = """
You are a junior loan underwriter AI. Evaluate the small business loan application against the bank's criteria.
Let's think step by step to create a clear evaluation report.

Application Data and Criteria:
{data}

**Evaluation Process:**
1.  **Check Years in Business:** Compare the applicant's years in business against the minimum requirement. State Pass or Fail.
2.  **Check Annual Revenue:** Compare the applicant's revenue against the minimum requirement. State Pass or Fail.
3.  **Check CIBIL Score:** Compare the owner's CIBIL score against the minimum requirement. State Pass or Fail.
4.  **Summary of Findings:** Briefly list the strengths and weaknesses of the application.
5.  **Final Recommendation:** Conclude with 'Proceed to Underwriting' or 'Reject'.
"""

_ = run_prompt(cot_loan_prompt, "Chain-of-Thought loan assessment", model=exploring_model,
               data=loan_application_data)

--- Chain-of-Thought loan assessment ---


**Evaluation Report for Loan Application: Global Exports Inc.**

1. **Check Years in Business:**  
   - Applicant's Years in Business: 2  
   - Minimum Required: 3  
   - **Result:** Fail (Applicant does not meet the minimum years in business requirement)

2. **Check Annual Revenue:**  
   - Applicant's Annual Revenue: ₹75,00,000  
   - Minimum Required: ₹50,00,000  
   - **Result:** Pass (Applicant meets the minimum annual revenue requirement)

3. **Check CIBIL Score:**  
   - Applicant's CIBIL Score: 680  
   - Minimum Required: 720  
   - **Result:** Fail (Applicant's CIBIL score is below the minimum requirement)

4. **Summary of Findings:**  
   - **Strengths:**  
     - Annual revenue is strong and exceeds the minimum threshold.  
   - **Weaknesses:**  
     - Business has been operational for only 2 years, less than the required 3 years.  
     - Owner's CIBIL score is 680, which is below the minimum required score of 720.

5. **Final Recommendation:**  
   - **Reject** due to failure to meet critical criteria on years in business and CIBIL score.

*Critique:* The response is transparent, auditable, and provides a clear, well-supported rationale for the decision.

> **Reasoning models change the arithmetic.** On a reasoning model such as `gpt-5-mini` you rarely need "let's think step by step" — the step-by-step thinking happens inside the model. The *structure* of this prompt still helps: it makes the reasoning you get back auditable, check by check.

## 8. Tree-of-Thought — compare strategies before recommending one

**Tree-of-Thought (ToT)** is an advanced technique that generalizes Chain-of-Thought. Instead of just one linear reasoning path, ToT encourages the model to explore *multiple* different lines of reasoning (branches of a tree). It can then self-evaluate these paths and choose the most promising one to pursue.

  * **How it works:** A full ToT implementation is complex and often requires an agentic framework with multiple LLM calls. However, we can simulate the *spirit* of ToT in a single prompt by asking the model to:
    1.  **Generate** multiple potential solutions or viewpoints.
    2.  **Evaluate** the pros and cons of each.
    3.  **Select** the best option based on that evaluation.

**Scenario:** A financial advisor is helping a client who is overwhelmed with multiple sources of debt. The goal is to propose and compare different concrete strategies for tackling the debt.

In [14]:
client_debt_profile = """
- Client: Priya
- Monthly Take-home Salary: ₹80,000
- Monthly Expenses (non-debt): ₹45,000
- Available for Debt Repayment: ₹35,000 per month
- Debts:
  1. Credit Card: ₹2,00,000 at 36% APR
  2. Personal Loan: ₹5,00,000 at 14% APR
  3. Car Loan: ₹3,00,000 at 9% APR
"""

naive_debt_prompt = "How can this client manage their debt? \n\n{profile}"
_ = run_prompt(naive_debt_prompt, "Naive debt advice", model=exploring_model,
               profile=client_debt_profile)

--- Naive debt advice ---


To help Priya manage her debt effectively, the goal is to minimize interest payments and become debt-free as quickly as possible while staying within her budget. Here's a step-by-step plan based on the information provided:

---

### Client Summary:
- **Monthly Take-home Salary:** ₹80,000  
- **Monthly Expenses (non-debt):** ₹45,000  
- **Available for Debt Repayment:** ₹35,000  
- **Debts:**
  1. Credit Card: ₹2,00,000 at 36% APR  
  2. Personal Loan: ₹5,00,000 at 14% APR  
  3. Car Loan: ₹3,00,000 at 9% APR  

---

### Step 1: Understand the Priority - Focus on High-Interest Debt First

- The credit card debt has the highest interest rate (36%), which means it grows fastest and costs the most.
- Next is the personal loan at 14%.
- The car loan has the lowest interest rate at 9%.

---

### Step 2: Debt Repayment Strategy - Avalanche Method

**Avalanche method** means paying off debts starting with the highest interest rate first while making minimum payments on the others.

---

### Step 3: Estimate Minimum Payments (Assumptions)

- Credit Card minimum payment: Typically 5% of balance → ₹10,000  
- Personal Loan EMI: Approximate (₹5,00,000 at 14% APR for 5 years) ≈ ₹11,900  
- Car Loan EMI: Approximate (₹3,00,000 at 9% APR for 5 years) ≈ ₹6,200  

**Total minimum payments:** ₹10,000 + ₹11,900 + ₹6,200 = ₹28,100  
**Available for extra repayment:** ₹35,000 - ₹28,100 = ₹6,900  

---

### Step 4: Allocate Payments

1. **Credit Card:** Pay ₹10,000 (minimum) + ₹6,900 (extra) = ₹16,900 per month  
2. **Personal Loan:** Pay ₹11,900 (minimum)  
3. **Car Loan:** Pay ₹6,200 (minimum)  

---

### Step 5: Timeline and Benefits

- Paying ₹16,900 monthly on the credit card will clear it faster, reducing the high interest burden.
- Once the credit card is paid off (approx 12-14 months), redirect that ₹16,900 + ₹11,900 (personal loan minimum) = ₹28,800 towards the personal loan.
- After the personal loan is cleared, use the full ₹35,000 to pay off the car loan faster.

---

### Step 6: Additional Tips

- Avoid accumulating new credit card debt.
- Consider negotiating lower interest rates or balance transfer offers for the credit card.
- Build an emergency fund once debts are under control to avoid future borrowing.
- Track expenses to see if ₹45,000 can be reduced to increase debt repayment capacity.

---

### Summary Table of Monthly Payments

| Debt           | Minimum Payment | Extra Payment | Total Payment |
|----------------|-----------------|---------------|---------------|
| Credit Card    | ₹10,000         | ₹6,900        | ₹16,900       |
| Personal Loan  | ₹11,900         | ₹0            | ₹11,900       |
| Car Loan       | ₹6,200          | ₹0            | ₹6,200        |
| **Total**      | ₹28,100         | ₹6,900        | ₹35,000       |

---

If Priya follows this plan, she will minimize interest costs and become debt-free in a structured and manageable way.

*Critique:* The advice is sound but lacks structure and fails to explore or compare alternative, well-known strategies.

#### **Improved "ToT-style" Prompt 👍**

This prompt instructs the model to generate and evaluate multiple distinct strategies, presenting a comprehensive analysis of the options.

In [15]:
tree_of_thought_debt_prompt = """
You are an expert financial advisor. For the client profile below, devise a debt management plan using the following structured thinking process.

Client Profile:
{profile}

1.  **Propose Three Distinct Strategies:** Generate three different debt repayment strategies. Name each one (e.g., 'Debt Avalanche', 'Debt Snowball', 'Loan Consolidation'). Briefly explain how each would work for this client.
2.  **Evaluate Each Strategy:** For each of the three strategies, list the primary 'Pro' (e.g., lowest total interest paid) and the primary 'Con' (e.g., may feel slower) for this specific client.
3.  **Recommend and Justify:** Based on your evaluation, select the single most suitable strategy and explain why it is the best recommendation for someone in this situation.
"""

_ = run_prompt(tree_of_thought_debt_prompt, "Tree-of-Thought debt plan", model=exploring_model,
               profile=client_debt_profile)

--- Tree-of-Thought debt plan ---


Certainly! Let's analyze Priya's debt situation and devise a structured debt management plan.

---

### Client Recap:
- **Monthly Take-home Salary:** ₹80,000  
- **Monthly Expenses (non-debt):** ₹45,000  
- **Available for Debt Repayment:** ₹35,000  
- **Debts:**
  1. Credit Card: ₹2,00,000 @ 36% APR  
  2. Personal Loan: ₹5,00,000 @ 14% APR  
  3. Car Loan: ₹3,00,000 @ 9% APR  

---

## 1. Propose Three Distinct Strategies

### Strategy A: Debt Avalanche  
**How it works:**  
Focus all available repayment funds on the debt with the highest interest rate first (Credit Card at 36%), while paying minimum amounts on the other loans. Once the highest interest debt is cleared, move to the next highest (Personal Loan at 14%), then finally the Car Loan (9%).  

---

### Strategy B: Debt Snowball  
**How it works:**  
Focus all extra repayment funds on the smallest debt first (Credit Card ₹2,00,000), regardless of interest rate, while paying minimums on the others. Once the smallest debt is cleared, move to the next smallest (Car Loan ₹3,00,000), then the largest (Personal Loan ₹5,00,000). This strategy emphasizes quick wins to build motivation.  

---

### Strategy C: Loan Consolidation & Refinance  
**How it works:**  
Attempt to consolidate the high-interest debts (Credit Card and Personal Loan) into a single loan with a lower interest rate, possibly by refinancing or taking a balance transfer at a lower rate. Continue paying the Car Loan separately. This reduces overall interest and simplifies payments.  

---

## 2. Evaluate Each Strategy

| Strategy             | Primary Pro                                                      | Primary Con                                                     |
|----------------------|-----------------------------------------------------------------|----------------------------------------------------------------|
| **Debt Avalanche**   | Minimizes total interest paid over time, saving money overall. | May take longer to see any debt fully paid off, which can be demotivating. |
| **Debt Snowball**    | Provides quick wins by paying off smaller debts first, boosting motivation. | Likely to pay more interest overall, especially on high-interest credit card debt. |
| **Loan Consolidation** | Potentially lowers interest rates and simplifies payments.     | Requires good credit score and may involve fees; not always feasible or immediate. |

---

## 3. Recommend and Justify

### Recommended Strategy: **Debt Avalanche**

**Justification:**  
- Priya’s highest interest debt (Credit Card at 36%) is significantly more expensive than the others. Paying this off first will drastically reduce the interest burden.  
- The available repayment amount (₹35,000) is substantial enough to aggressively pay down the credit card within a reasonable timeframe, providing a balance between motivation and cost savings.  
- Although the Debt Snowball offers psychological benefits, the high interest rate on the credit card means that delaying its repayment will cost Priya a lot more in interest.  
- Loan consolidation is worth exploring but may not be immediately available or feasible, and the personal loan and car loan have relatively moderate interest rates.  
- After clearing the credit card, Priya can then focus on the personal loan and finally the car loan, ensuring the most cost-effective path to becoming debt-free.

---

### Additional Recommendations:

- **Minimum Payments:** Ensure minimum payments are made on all loans to avoid penalties.  
- **Emergency Fund:** Maintain a small emergency fund to avoid further debt in case of unexpected expenses.  
- **Review After Credit Card:** Once the credit card is paid off, consider if refinancing the personal loan is beneficial.  

---

If you want, I can also provide a rough timeline and interest cost estimate for the Debt Avalanche strategy. Would you like that?

*Critique:* This response provides a strategic, multi-faceted analysis that empowers the client and advisor to make an informed decision based on clear trade-offs.

## 9. Structured output with Pydantic — the headline skill ⭐

So far every answer was **free text**. Production systems need **structured, machine-readable** data they can route, store, and act on. Asking a model to "return JSON" is fragile (it adds prose, breaks quotes, drifts schema). Instead we define a **Pydantic schema** and let the provider **guarantee** the output matches it.

**Scenario — support triage.** Every inbound message must land in a fixed schema so the case system can auto-route it to the right queue.

In [16]:
class SupportTicket(BaseModel):
    """Structured triage for any inbound customer message."""
    category: Literal["billing", "technical", "complaint", "inquiry", "fraud_or_safety", "cancellation"] = Field(
        description="Primary intent of the message")
    urgency: Literal["low", "medium", "high", "critical"] = Field(description="How fast a human must act")
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    industry: str = Field(description="Best-guess industry, e.g. 'banking', 'healthcare', 'e-commerce'")
    entities: List[str] = Field(description="Key entities: order ids, amounts, names, dates")
    summary: str = Field(description="One-sentence summary")
    recommended_action: str = Field(description="Suggested next step for the support agent")


# The LangChain way: the model returns a validated SupportTicket object, not text.
reasoning_model = ChatOpenAI(model="gpt-5-mini")        # reasoning model: no temperature
triage_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the customer message into the SupportTicket schema."),
    ("human", "{message}"),
])
triage_chain = triage_prompt | reasoning_model.with_structured_output(SupportTicket)

message_text = ("My credit card was charged ₹18,250 TWICE for the same order #A4821 this morning. "
                "I need one charge refunded today — this is unacceptable.")
ticket = triage_chain.invoke({"message": message_text})
print(type(ticket).__name__, "->")
print(ticket.model_dump_json(indent=2))
print("\nRoute on a field, no string parsing:", "ESCALATE" if ticket.urgency in ("high", "critical") else "queue")

SupportTicket ->
{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "Order #A4821",
    "₹18,250",
    "charged twice",
    "this morning",
    "credit card"
  ],
  "summary": "Customer reports being charged twice (₹18,250 each) for order #A4821 and demands one charge refunded today.",
  "recommended_action": "Immediately verify payment records for order #A4821 and confirm the duplicate charge. If confirmed, process a refund for one ₹18,250 charge and send the customer a same-day confirmation and apology. If additional transaction details are needed, request the card's last 4 digits and transaction IDs; escalate to billing/finance if the refund cannot be completed immediately."
}

Route on a field, no string parsing: ESCALATE


In [17]:
# The same call with the raw OpenAI SDK — for one call to one provider, this is all you need.
from openai import OpenAI

openai_client = OpenAI()
response = openai_client.responses.parse(
    model="gpt-5-mini",
    input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
           {"role": "user", "content": message_text}],
    text_format=SupportTicket,
)
print("Raw SDK ->", response.output_parsed.category, "/", response.output_parsed.urgency)

Raw SDK -> billing / high


### When the output does not match — validation failure, then retry

A schema is only useful if a **bad** payload is stopped before it reaches the case system. Pydantic does the
stopping; a bounded retry gives the model another chance.

In [18]:
# A payload that breaks the schema: "urgency" must be one of low / medium / high / critical.
malformed_payload = ('{"category": "billing", "urgency": "extreme", "sentiment": "angry", '
                     '"industry": "banking", "entities": ["18,250"], '
                     '"summary": "Duplicate card charge.", "recommended_action": "Refund one charge."}')
try:
    SupportTicket.model_validate_json(malformed_payload)
except ValidationError as validation_error:
    print("Rejected before it reached any downstream system:")
    for problem in validation_error.errors():
        print(f"  field {problem['loc']}: {problem['msg']}")

# OpenAI enforces the schema on its side, so a live mismatch is rare here. Providers that only
# *prompt* for JSON mismatch far more often. The defence is the same everywhere: validate, and
# retry a bounded number of times. `.with_retry()` works on any Runnable — Lab 2.2 returns to it.
resilient_triage_chain = triage_chain.with_retry(
    retry_if_exception_type=(ValidationError, OutputParserException), stop_after_attempt=3)
retried_ticket = resilient_triage_chain.invoke(
    {"message": "I got an OTP I never requested and someone tried to log into my account. Please block it NOW."})
print("\nWith a bounded retry in place ->", retried_ticket.category, "/", retried_ticket.urgency)

Rejected before it reached any downstream system:
  field ('urgency',): Input should be 'low', 'medium', 'high' or 'critical'



With a bounded retry in place -> fraud_or_safety / critical


## 10. Two applied banking scenarios

### Scenario 1 — Summarising a customer loan inquiry

A loan officer needs a quick summary of a long customer email to assess eligibility at a glance.

### **Bad Prompt Example 👎**

This prompt is too vague. It doesn't specify the output format or the key information to extract.

In [19]:
customer_email = """
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.
"""

bad_prompt_1 = "Summarize this email: {customer_email}"
_ = run_prompt(bad_prompt_1, "Scenario 1: bad prompt", customer_email=customer_email)

--- Scenario 1: bad prompt ---


Priya Sharma, a loyal customer of over 10 years and a software developer earning ₹18,00,000 annually, is inquiring about obtaining a ₹5,00,000 personal loan for a family wedding. She has a good credit score of around 780, no existing loans with the bank, and a credit card with a ₹2,00,000 limit. She requests information on the next steps.

*Critique:* The summary is okay, but it's unstructured and mixes all the information. A loan officer would have to read through the paragraph to find specific data points.

### **Good Prompt Example 👍**

This prompt is precise. It assigns a **role** (Loan Officer Assistant), provides clear **context**, specifies the **task**, and sets **constraints** on the output format (JSON).

In [20]:
good_prompt_1 = """
You are a highly efficient AI assistant for a bank's loan officer.
Your task is to extract key information from a customer's loan inquiry email and present it in a structured JSON format.

**Email Content:**
---
{customer_email}
---

**Instructions:**
1. Extract the following fields: customer_name, loan_amount, loan_purpose, annual_salary, cibil_score, and existing_liabilities.
2. If a specific piece of information is not mentioned, use "N/A".
3. The output must be a valid JSON object. Do not add any introductory text or explanations.
"""

_ = run_prompt(good_prompt_1, "Scenario 1: good prompt", customer_email=customer_email)

--- Scenario 1: good prompt ---


```json
{
  "customer_name": "Priya Sharma",
  "loan_amount": "₹5,00,000",
  "loan_purpose": "family wedding",
  "annual_salary": "₹18,00,000",
  "cibil_score": 780,
  "existing_liabilities": "Credit card with ₹2,00,000 limit"
}
```

*Critique:* This response is perfect. The JSON format is machine-readable and easy for the loan officer to quickly parse. All key information is clearly labeled.

> **Connect it to section 9.** "Return valid JSON" in a prompt is a *request*. Swap it for `with_structured_output(...)` and a Pydantic model, and it becomes a *guarantee* your code can rely on.

### Scenario 2 — A customer message for a failed transaction

A customer's online payment failed. The bank needs to send a clear, empathetic, and helpful notification.

### **Bad Prompt Example 👎**

This prompt lacks tone guidance and doesn't instruct the model on what helpful actions to suggest.

In [21]:
transaction_details = {
    "customer_name": "Mr. Verma",
    "amount": "₹3,500",
    "merchant": "Flipkart",
    "reason": "Incorrect CVV"
}

bad_prompt_2 = "Tell {customer_name} that their payment of {amount} to {merchant} failed because of {reason}."
_ = run_prompt(bad_prompt_2, "Scenario 2: bad prompt", **transaction_details)

--- Scenario 2: bad prompt ---


Sure! Here's a message you can send to Mr. Verma:

---

Dear Mr. Verma,

We would like to inform you that your payment of ₹3,500 to Flipkart was unsuccessful due to an incorrect CVV entered. Please verify your card details and try again.

If you need any assistance, feel free to reach out.

Thank you.

---

Would you like me to make it more formal or casual?

*Critique:* This response is blunt, robotic, and unhelpful. It could cause customer anxiety and doesn't offer a solution.

### **Good Prompt Example 👍**

This prompt uses the **role-playing** technique, specifies the desired **tone** (empathetic and professional), and provides a clear **structure** with actionable next steps. This is a simple form of **few-shot prompting** by providing an example of the desired output style within the instructions.

In [22]:
good_prompt_2 = """
You are a Customer Service Bot for SecureBank.
Your task is to draft an SMS notification to a customer about a failed transaction.

**Tone:** Empathetic, professional, and reassuring.

**Instructions:**
1. Address the customer by name.
2. Clearly state that the transaction could not be processed.
3. Mention the amount and merchant.
4. Briefly and simply state the reason for the failure.
5. Provide a clear, immediate action the customer can take (e.g., "please try again, ensuring the details are correct").
6. End on a helpful note, providing a support channel.

**Customer and Transaction Details:**
- Customer Name: {customer_name}
- Amount: {amount}
- Merchant: {merchant}
- Reason: {reason}

Draft the SMS message.
"""

_ = run_prompt(good_prompt_2, "Scenario 2: good prompt", **transaction_details)

--- Scenario 2: good prompt ---


Dear Mr. Verma, your transaction of ₹3,500 at Flipkart could not be processed due to an incorrect CVV. Please try again, ensuring the CVV entered is correct. For assistance, contact SecureBank support at 1800-123-456. We're here to help.

*Critique:* This response is excellent. It's polite, explains the problem clearly, offers an immediate solution, and provides a way to get more help, building customer trust.

## 11. Choosing a pattern

| Pattern | Reach for it when | What it costs you |
|---|---|---|
| **Zero-shot** | Simple tasks on capable models | Nothing — but the format is whatever the model chooses |
| **Few-shot** | The format is unusual, or accuracy matters — it locks format and decision boundaries the model can't infer from instructions | Longer prompts; examples to maintain |
| **Directional stimulus** | You need specific aspects of the input — e.g. "focus on the negative factors" | A narrower answer, by design |
| **Chain-of-Thought** | Multi-step reasoning must be auditable — each check visible | Longer outputs, more latency and tokens |
| **Tree-of-Thought** | Ambiguous, multi-strategy decisions — explore several options, then pick | The longest outputs; worth it only when the choice matters |
| **Structured output** | Anything a program consumes next — routing, storage, dashboards | A schema to design and version |

### The five habits behind every good prompt (from Scenarios 1–2)
1.  **Be Specific:** Vague prompts lead to vague answers. Tell the model exactly what you want.
2.  **Assign a Role:** Telling the LLM to act as a "Loan Officer" or "Marketing Expert" primes it for better, context-aware responses.
3.  **Define the Format:** Don't leave the output structure to chance. Explicitly ask for JSON, a list, an email, or a table.
4.  **Set the Tone:** The same information can be delivered in many ways. Specify the tone (e.g., formal, empathetic, persuasive) to match the use case.
5.  **Provide Context:** The LLM doesn't know what you know. Give it all the necessary background information to complete the task successfully.

> ⏭️ **After the session.** This section is not run live in the workshop. Work through it afterwards, top to bottom.

## 12. Your turn — challenge

Now, apply what you've learned. For the following scenario, write a precise "good" prompt yourself.

**Scenario:** A bank wants to generate a marketing email to promote its new "GlobalTravel" credit card to existing customers who are identified as frequent travelers.

**Key Features of the Card:**

  * Zero foreign transaction fees.
  * Complimentary airport lounge access (4 per year).
  * 10,000 bonus reward points on signup.
  * Annual fee: ₹2,500.

**Your task:** write `challenge_prompt` with a **role**, the **target audience**, the **tone**, a clear
**structure** and a `[Customer Name]` placeholder — then run it. Stretch: make the output a Pydantic
`MarketingEmail(subject, greeting, body, call_to_action)` instead of free text.

In [23]:
# Your code here!
#
# challenge_prompt = """ ... your prompt ... """
# _ = run_prompt(challenge_prompt, "Challenge: marketing email")

## 13. Conclusion & key takeaways

| You practised | Why it matters |
|---|---|
| Messages, `ChatPromptTemplate`, `MessagesPlaceholder` | Every chain, retriever and agent in this workshop starts here |
| Tokens and context windows | Cost and latency scale with tokens — and a window has a limit |
| Five patterns, one example each | Pick the pattern for the task instead of rewriting prompts blindly |
| Structured output with Pydantic | Output a program can trust — the backbone of tool calls, routing and graph state |
| Validation failure and a bounded retry | Bad output is stopped, not stored |

**Next →** Module 2, Lab 2.1 — **LangChain Fundamentals**: composing these prompts into LCEL
chains, parsing, streaming and running branches in parallel.